In [ ]:
"""
Objetivo: Estimar a Persistência/Meia-Vida univariada de cada carteira de crédito e aplicar
o método de Projeções Locais (Jordà, 2005) para mapear o impacto no desemprego.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.ar_model import ar_select_order, AutoReg
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==========================================
def carregar_dados_lp(arquivo_csv: str) -> pd.DataFrame:
    """Carrega os dados e preserva a amostra máxima disponível (desde 2000)."""
    print("[*] Carregando a base de dados limpa...")
    df = pd.read_csv(arquivo_csv, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    
    print(f"[*] Amostra ampliada: {df_model.index[0].date()} a {df_model.index[-1].date()}")
    print(f"[*] Total de observações: {len(df_model)} trimestres")
    return df_model

CARTEIRAS = {
    'inad_consumidor': 'Crédito ao Consumidor',
    'inad_hipotecas': 'Hipotecas',
    'inad_cartoes': 'Cartões de Crédito'
}

# ==========================================
# 2. PERSISTÊNCIA E MEIA-VIDA (AR UNIVARIADO)
# ==========================================
def calcular_meia_vida(df: pd.DataFrame, dicionario_carteiras: dict):
    """Calcula a persistência e meia-vida (AR(p)) selecionada via BIC."""
    print("\n--- Persistência e Meia-Vida (Choque Interno) ---")
    for col, nome in dicionario_carteiras.items():
        serie = df[col].dropna()
        lags_otimos = ar_select_order(serie, maxlag=4, ic='bic').ar_lags
        modelo_ar = AutoReg(serie, lags=lags_otimos).fit()
        
        soma_ar = np.sum(modelo_ar.params[1:])
        if 0 < soma_ar < 1:
            meia_vida = np.log(0.5) / np.log(soma_ar)
            print(f"[{nome}] Ordem: AR({len(lags_otimos)}) | Persistência: {soma_ar:.3f} | Meia-vida: {meia_vida:.2f} trim.")
        else:
            print(f"[{nome}] Ordem: AR({len(lags_otimos)}) | Série apresenta raiz unitária ou não reverte à média.")

# ==========================================
# 3. PROJEÇÕES LOCAIS DE JORDÀ (ESTIMAÇÃO E GRÁFICOS)
# ==========================================
def executar_projecoes_locais(df: pd.DataFrame, dicionario_carteiras: dict, h_max: int = 12):
    """
    Estima os vetores de Projeções Locais usando regressões HAC (Newey-West) 
    e exporta os gráficos de Funções de Resposta ao Impulso (IRF).
    """
    print("\n[*] Estimando Projeções Locais (Horizonte: 12 trimestres)...")
    
    # Resposta (Desemprego) em primeira diferença
    y = df['desemprego'].diff().dropna()
    
    for col, nome in dicionario_carteiras.items():
        # Choque contemporâneo também em diferença
        x = df[col].diff().dropna()
        
        betas, ic_inf, ic_sup = [], [], []
        
        for h in range(h_max + 1):
            y_futuro = y if h == 0 else y.shift(-h)
            
            dados = pd.DataFrame({
                'y_futuro': y_futuro, 'x_choque': x,
                'y_lag1': y.shift(1), 'x_lag1': x.shift(1)
            }).dropna()
            
            X = sm.add_constant(dados[['x_choque', 'y_lag1', 'x_lag1']])
            Y = dados['y_futuro']
            
            resultado_hac = sm.OLS(Y, X).fit(cov_type='HAC', cov_kwds={'maxlags': h + 1})
            
            betas.append(resultado_hac.params['x_choque'])
            ic_inf.append(resultado_hac.conf_int(alpha=0.05).loc['x_choque', 0])
            ic_sup.append(resultado_hac.conf_int(alpha=0.05).loc['x_choque', 1])
            
        # Exportar gráfico
        _plotar_irf_lp(nome, list(range(h_max + 1)), betas, ic_inf, ic_sup, h_max)

def _plotar_irf_lp(nome: str, horizonte: list, irf: list, ic_inf: list, ic_sup: list, h_max: int):
    """Função auxiliar para salvar os gráficos da Projeção Local."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(horizonte, irf, color='firebrick', linewidth=2, label='Resposta Estimada (LP)')
    ax.fill_between(horizonte, ic_inf, ic_sup, color='red', alpha=0.15, label='Intervalo de Confiança (95%)')
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    
    ax.set_ylabel('Impacto no Desemprego (p.p.)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Horizonte (Trimestres)', fontsize=10, fontweight='bold')
    ax.legend(loc='best', fontsize=9, frameon=False)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xticks(range(0, h_max + 1, 2))
    plt.tight_layout()
    
    nome_arquivo = nome.lower().replace(' ', '_').replace('ç', 'c').replace('õ', 'o').replace('é', 'e')
    arquivo_saida = f'02_irf_lp_{nome_arquivo}.png'
    plt.savefig(arquivo_saida, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[*] Gráfico exportado: '{arquivo_saida}'")

if __name__ == "__main__":
    df_model = carregar_dados_lp('base_macro_fred_limpa.csv')
    calcular_meia_vida(df_model, CARTEIRAS)
    executar_projecoes_locais(df_model, CARTEIRAS, h_max=12)